In [ ]:
import requests
import time
from datetime import datetime, timedelta
import pandas as pd

In [ ]:
BASE_SEARCH_URL = "https://api.hh.ru/vacancies"
HEADERS = {"User-Agent": "ResearchScript/1.0 (anna_zagurskaya14@mail.ru)"}
MAX_PER_DAY = 2000

def daterange(start_date, end_date):
    for n in range(int((end_date - start_date).days)):
        yield start_date + timedelta(n)

start_date = datetime(2025, 12, 1)
end_date = datetime(2026, 1, 12)

all_ids = set()

for single_date in daterange(start_date, end_date):
    date_str = single_date.strftime("%Y-%m-%d")
    print(f"\nДень {date_str} — старт")

    page = 0
    day_count = 0
    per_page = 100

    while True:
        params = {
            "date_from": date_str,
            "date_to": date_str,
            "only_with_salary": True,
            "per_page": per_page,
            "page": page
        }

        response = requests.get(BASE_SEARCH_URL, params=params, headers=HEADERS)
        response.raise_for_status()
        data = response.json()

        if page == 0:
            print(f"   Найдено по API: {data.get('found')}")

        items = data.get("items", [])
        if not items:
            break

        for item in items:
            all_ids.add(item["id"])
            day_count += 1
            if day_count >= MAX_PER_DAY:
                break

        if day_count >= MAX_PER_DAY or page >= data["pages"] - 1:
            break

        page += 1
        time.sleep(0.2)

    print(f"   Собрано за день: {day_count}")

print(f"\nВсего уникальных вакансий за период: {len(all_ids)}")

pd.DataFrame(all_ids).rename(columns={0:'id'}).to_excel('vacancies_ids.xlsx', index=False)

In [ ]:
df = pd.read_excel('vacancies_ids.xlsx')
ids_list = df['id'].astype(str).tolist()
len(ids_list)

In [ ]:
BIG_CHUNK_SIZE = 10000
BATCH_SIZE = 1000
SLEEP_BETWEEN_REQUESTS = 0.5
SLEEP_BETWEEN_BATCHES = 60

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

def flatten_dict(d, parent_key='', sep='_'):
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep))
        elif isinstance(v, list):
            items[new_key] = json.dumps(v, ensure_ascii=False)
        else:
            items[new_key] = v
    return items

total_ids = len(ids_list)
print(f"Всего вакансий к обработке: {total_ids}")

for TARGET_CHUNK in range(1, 9):
    big_start = BIG_CHUNK_SIZE * (TARGET_CHUNK - 1)
    big_end = min(big_start + BIG_CHUNK_SIZE, total_ids)
    ids_big_chunk = ids_list[big_start:big_end]

    print(f"\nChunk {TARGET_CHUNK}: [{big_start}–{big_end})")

    all_batches_dfs = []
    batch_number = 0

    for batch_ids in chunks(ids_big_chunk, BATCH_SIZE):
        batch_number += 1
        batch_start = big_start + (batch_number - 1) * BATCH_SIZE
        batch_end = min(batch_start + len(batch_ids), big_end)

        print(f"\nБатч {batch_number} (в chunk {TARGET_CHUNK}) [{batch_start}–{batch_end}) — {len(batch_ids)} вакансий")

        vacancies_list = []

        for idx, vac_id in enumerate(batch_ids, start=1):
            global_idx = batch_start + idx
            url = f"{BASE_VACANCY_URL}/{vac_id}"

            try:
                response = requests.get(url, headers=HEADERS, timeout=10)
                if response.status_code == 403:
                    print(f"chunk {TARGET_CHUNK} | batch {batch_number} | 403 ошибка – {vac_id}")
                    continue

                response.raise_for_status()
                data = response.json()
                flat_data = flatten_dict(data)
                vacancies_list.append(flat_data)

                print(f"   chunk {TARGET_CHUNK} | batch {batch_number} | {idx}/{len(batch_ids)} | global {global_idx}/{total_ids} → {vac_id}")
                time.sleep(SLEEP_BETWEEN_REQUESTS)

            except requests.exceptions.RequestException as e:
                print(f"   chunk {TARGET_CHUNK} | batch {batch_number} | {vac_id} → {e}")
                continue

        if vacancies_list:
            df_batch = pd.DataFrame(vacancies_list)
            all_batches_dfs.append(df_batch)
            print(f"Батч {batch_number} сохранён ({len(df_batch)} строк)")

        time.sleep(SLEEP_BETWEEN_BATCHES)

    if all_batches_dfs:
        df_chunk = pd.concat(all_batches_dfs, ignore_index=True, sort=False)
        filename = f"vacancies_{big_start}_{big_end}.xlsx"
        df_chunk.to_excel(filename, index=False)
        print(f"\nФайл сохранён: {filename}")
        print(f"Размер DataFrame: {df_chunk.shape}")

    print(f"\nСбор вакансий (chunk {TARGET_CHUNK}) завершён")